# Generator Section
Functions that return and object that can be iterated over.
Generate the items one at a time and only when you ask for it. They are more efficient in memory storage

In [ ]:
#Example generator 
def myGenerator():
    yield 1 
    yield 2
    yield 3


gen = myGenerator()
gen

In [ ]:
#we can run a for loop to get all values in the function 
# for i in gen:
#     print(i)

In [ ]:
#OR we can use the next() to get the next value in the function 
#if you run this block again, it will go to the next line
#after running it the number of times of the 'yield', then it stops the iteration
value = next(gen)
value

In [ ]:
#USe them as input to other funcitons that take iterables
sorted(gen)

In [ ]:
def countdown(num):
    print ("starting")
    while num > 0: 
        yield num
        num -= 1

cd = countdown(5)
val = next(cd)
val

In [ ]:
print(next(cd))
print(next(cd))
print(next(cd))

In [ ]:
print(next(cd))

In [ ]:
import sys
#They save memory when working with large data
def firstn(n):
    nums = []
    num = 0
    while num < n:
        nums.append(num)
        num += 1
    return nums

#This is the generator object of the above function
def firstnGen(n):
    num = 0
    while num < n:
        yield num
        num += 1

sys.getsizeof(firstn(1000))


In [ ]:
#the size of the generation object with a list of 10 or 1000 numbers remains the same
sys.getsizeof(firstnGen(1000))


In [ ]:
#another practice example: fibonacci sequence
def fibonacci(limit):
    # 0, 1, 1, 2, 3, 5, 8, 13....
    firstVal = 0
    secondVal = 1
    while firstVal < limit:
        yield firstVal
        firstVal, secondVal = secondVal, firstVal + secondVal

fib = fibonacci(30)
for i in fib: 
    print(i)

In [ ]:
#
newGenerator = (i for i in range(10) if i % 2 == 0)
print(sys.getsizeof(newGenerator))
list(newGenerator)

# Threading vs multiprocessing section 
Run code in parallel, difference between porcesses and threads, advantages and disadvantages 
how to run multiple threads

Process: An instance of a program (eg python interpretor or firefox)
- /+ takes advantage of multiple CPUs and cores
- /+ Separate memor space -> memory is not shared between processes
- /+ Great for CPU-bound processing 
- /+ new process is stated independently from other processes
- /+ processes are interruptable/killable
- /+ One GIL (global interpreter log) for each process -> Avoids GIL limitation 

- /- Heavywheight
- /- starting a process is slower than starting a thread
- /- more memory
- /- IPC (inter-process communicaiton) is more complicated

Thread: An entity within a process that can be scheduled for execuation
a process can spawn mulitple thread

- /+ they are lighweight
- /+ all threads within a process share the same memory
- /+ starting a thread is faster than starting a process
- /+ great for I/O bound tasks

- /- Threading is limired by GIL: only one thread at a time
- /- no effect on CPU bound tasks
- /- Not interruptable or killable
- /- careful with race conditions - two or more threads are trying to modify same vairables 

GIL: Global interpreter lock
- A lock that allows only one thread at a time to execute in Python
- Needed in CPython becasue memory management is not thread-safe

- Avoid:
    - use multiprocessing
    - use a different, free-threaded python implementation (Jython, IronPython)
    - use Python as a wrapper for third part libraires (C/C++) -> numpy, scipy

In [ ]:
from multiprocessing import Process
import os
import time

processes = []
numOfProcesses = os.cpu_count()

numOfProcesses
def squareNum():
    for i in range(100):
        i*i
        time.sleep(0.1)

In [ ]:

for i in range(numOfProcesses):
    p = Process(target=squareNum)
    processes.append(p)

#start the process
for p in processes:
    p.start()

# join the process
for p in processes:
    p.join()

print('end main')
#this take the 10secs that were set in the sleep function. 
# This runs two process based on the number of CPUs for notebook, one at a time and then ends them both one after the other

# Multi threading


In [ ]:
from threading import Thread

#Similar to the process item above, the code is basically the same
threads =[]
numThreads = 10

#create the threads
for i in range(numThreads):
    #main fucntion to run in the thread - target function
    t = Thread(target=squareNum)
    threads.append(t)

#start the process
for t in threads:
    t.start()

# join the process
for t in threads:
    t.join()

print('end main')

# Threading Section

Locks for race condition, queue for data exchanges etc

In [ ]:
#share data between threads as they live in the same memory 
#due to race condition for thread1 and thread2 trying to change the same memory variable 'local_cpy'
#the end value with out a 'lock' condition would be 1

from threading import Lock

#global variable
database_val = 0

def increase(lock):
    global database_val

    # first thread locked the state and modified the value, it wont swithc to thread2 until the memory is released from lock
    lock.acquire() 
    local_cpy = database_val
    # some processing
    local_cpy += 1
    time.sleep(0.05)

    database_val = local_cpy
    lock.release()

lock = Lock()
print("start value", database_val)

thread1 = Thread(target=increase, args=(lock,))
thread2 = Thread(target=increase, args=(lock,))

thread1.start()
thread2.start()

thread1.join()
thread2.join()

print("end value", database_val)
    

In [ ]:
#using queues in python for multi threaded and multi processing environemnt

from queue import Queue
from threading import current_thread

#first in first out 
q = Queue()

def worker(q, lock):
    while True: 
        value = q.get()
        #processing
        with lock:
            print(f'in {current_thread().name} got {value}')
            q.task_done()


for i in range(numThreads):
    thread = Thread(target=worker, args=(q,lock))
    #daemon thread dies when the main thread dies. So it will end the tasks that are going on
    thread.daemon = True 
    thread.start()

for i in range (1,11):
    q.put(i)

q.join()

print('end task')